# Fotony v pásmech Sentinelu-2C

[![Otevřít v Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/rope_and_photons_brno_cs.ipynb)

**OpenSkyLab · notebook k laboratoři** [Švihadlo a fotony](https://openskylab.space/edu_viz_rope_and_photons.html) · [English version](rope_and_photons_brno_en.ipynb)

V laboratoři jsi sledoval vlnu na laně. Její rychlost $v$, vlnovou délku
$\lambda$ (vzdálenost mezi sousedními vrcholy) a frekvenci $f$ (počet kmitů
za sekundu) spojuje vztah

$$
v = \lambda f.
$$

**Pro světlo ve vakuu platí stejný vztah**, s rychlostí světla $c$. Světlo
přenáší energii ve fotonech. Z vlnové délky určíme frekvenci a energii
jednoho fotonu:

$$
f = \frac{c}{\lambda}, \qquad E = hf.
$$

$h$ je Planckova konstanta; energii budeme vyjadřovat v elektronvoltech (eV),
jednotce energie. **Kratší vlnová délka znamená vyšší frekvenci a větší energii
fotonu.** Při stejné frekvenci znamená více fotonů dopadajících na stejnou
plochu za sekundu více přenesené energie; energie jednotlivého fotonu se nemění.
Tady to spočítáš pro skutečná pásma družice Sentinel-2C.

**Cíl:** z metadat skutečné scény Sentinel-2 spočítat frekvenci, energii fotonu a tok fotonů v každém pásmu a odvodit z nich, jaký detektor které pásmo potřebuje.

**Data:** metadata produktu Sentinel-2C L1C, dlaždice 33UXQ (okolí Brna), 5. 2. 2025 (`MTD_MSIL1C.xml` z veřejného archivu Google Cloud): střední vlnové délky, spektrální citlivost pásem a sluneční ozáření nad atmosférou. Stahuje se asi 50 kB.

**Úkoly:** šest. Odpovědi piš do buněk *Tvoje odpověď* pod úkoly. Řádky označené `# ◀ parametr` můžeš měnit a buňku spustit znovu (Shift + Enter).

**Čas:** asi 40 min. Potřebuješ Google účet (Colab); nic se neinstaluje do počítače. Buňky spouštěj popořadě shora dolů.

## 1. Příprava a data

**Povinná buňka, spusť ji jako první.** Načte knihovny a fyzikální konstanty. Kód je sbalený; zobrazíš ho pomocí **Show code**.

In [ ]:
# @title Příprava (spusť jako první) {display-mode: "form"}
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

# h: Planckova konstanta, c: rychlost světla,
# e: elementární náboj, k: Boltzmannova konstanta (vše v SI)
from scipy.constants import c, e, h, k

> **💻 Co dělá kód:** Stáhne soubor metadat (XML) jedné skutečné scény Sentinelu-2 L1C.

In [ ]:
# metadata jedné skutečné scény Sentinel-2C L1C (dlaždice 33UXQ, 5. 2. 2025)
URL = (
    "https://storage.googleapis.com/gcp-public-data-sentinel-2/tiles/33/U/XQ/"
    "S2C_MSIL1C_20250205T095231_N0511_R079_T33UXQ_20250205T115121.SAFE/MTD_MSIL1C.xml"
)
xml = requests.get(URL, timeout=60).text
print(f"staženo {len(xml) / 1000:.0f} kB")

> **💻 Co dělá kód:** Z XML vytáhne pro každé pásmo název, střední vlnovou délku, spektrální citlivost po 1 nm a sluneční ozáření nad atmosférou E0. Ukáže tabulku.

In [ ]:
# z XML vytáhneme pro každé pásmo: název, střední vlnovou délku, spektrální citlivost a sluneční ozáření E0
rows, srf = [], {}
for m in re.finditer(
    r'<Spectral_Information bandId="(\d+)" physicalBand="(\w+)">(.*?)</Spectral_Information>',
    xml,
    re.S,
):
    bid, name, body = int(m.group(1)), m.group(2), m.group(3)
    lmin = float(re.search(r'<MIN unit="nm">([\d.]+)', body).group(1))
    vals = np.array(re.search(r"<VALUES>(.*?)</VALUES>", body).group(1).split(), float)
    srf[name] = (lmin + np.arange(len(vals)), vals)  # krok 1 nm
    e0 = float(
        re.search(rf'<SOLAR_IRRADIANCE bandId="{bid}"[^>]*>([\d.]+)', xml).group(1)
    )
    rows.append(
        {
            "band": name,
            "lambda_nm": float(
                re.search(r'<CENTRAL unit="nm">([\d.]+)', body).group(1)
            ),
            "E0_W_m2_um": e0,
        }
    )
bands = pd.DataFrame(rows).set_index("band")
bands

> **💻 Co dělá kód:** Vykreslí spektrální citlivost všech pásem podle vlnové délky. Nic nepočítá.

In [ ]:
# spektrální citlivost pásem: jak silně detektor reaguje na jednotlivé vlnové délky
fig, ax = plt.subplots(figsize=(11, 3.2))
for name, (wl, r) in srf.items():
    ax.plot(wl, r, lw=1)
    ax.text(bands.lambda_nm[name], 1.03, name, ha="center", fontsize=8)
ax.set(
    xlabel="vlnová délka (nm)",
    ylabel="relativní citlivost",
    ylim=(0, 1.12),
    title="Sentinel-2C MSI: spektrální citlivost pásem",
)
plt.show()

## 2. Vlnová délka a frekvence

> **💻 Co dělá kód:** Ze střední vlnové délky každého pásma spočítá frekvenci f = c/λ a pro kontrolu součin λ · f. Ukáže tabulku.

In [ ]:
lam = bands.lambda_nm * 1e-9  # m
bands["f_THz"] = c / lam / 1e12  # f = c / λ
bands["lambda_x_f"] = lam * bands.f_THz * 1e12
bands[["lambda_nm", "f_THz", "lambda_x_f"]].round(1)

> **✏️ Úkol 1.** Sloupec `lambda_x_f` je u všech pásem stejný. Co vyjadřuje a čemu odpovídá na laně v laboratoři? Kolikrát vyšší frekvenci má pásmo B2 (modrá) než B12 (SWIR 2)?

*Tvoje odpověď:* 

## 3. Energie jednoho fotonu

Energii jednoho fotonu určuje Planckův–Einsteinův vztah:

$$
E = hf = \frac{hc}{\lambda}.
$$

Používáme $h \approx 6{,}626\times10^{-34}\ \mathrm{J\,s}$,
$c \approx 3{,}00\times10^8\ \mathrm{m\,s^{-1}}$ a vlnovou délku
$\lambda$ v metrech, takže výsledek vyjde v joulech.
Protože $1\ \mathrm{eV} = 1{,}602\,176\,634\times10^{-19}\ \mathrm{J}$,
převedeme energii na eV vydělením touto hodnotou. V kódu ji poskytuje
číselná hodnota `e`, elementárního náboje v coulombech, ze `scipy.constants`.

> **💻 Co dělá kód:** Spočítá energii fotonu ve středu každého pásma v eV, vykreslí ji jako sloupcový graf a vypíše poměr energie modrého a červeného fotonu.

In [ ]:
bands["E_photon_eV"] = h * c / lam / e  # E = h·f = h·c/λ, převedeno na elektronvolty
ax = bands.E_photon_eV.plot.bar(figsize=(9, 3), color="tab:purple")
ax.set(ylabel="energie fotonu (eV)", title="Energie jednoho fotonu ve středu pásma")
plt.show()
print(
    f"modrý (B2) / červený (B4) foton: {bands.E_photon_eV['B2'] / bands.E_photon_eV['B4']:.2f} ×"
)

> **✏️ Úkol 2.** Ověř poměr energií modrého a červeného fotonu jen z vlnových délek, bez konstant. Proč stačí poměr λ? Jak se změní energie jednoho fotonu, když je modré světlo dvakrát jasnější?

*Tvoje odpověď:* 

## 4. Od energie záření k toku fotonů

Metadata uvádějí spektrální sluneční ozářenost $E_0$ nad atmosférou
v $\mathrm{W\,m^{-2}\,\mu m^{-1}}$. Vydělením energií jednoho fotonu
v joulech získáme počet fotonů dopadajících za sekundu na metr čtvereční
na jednotku vlnové délky, zde na mikrometr:

$$
\Phi_\lambda = \frac{E_0}{E}.
$$

$\Phi_\lambda$ má jednotku $\mathrm{s^{-1}\,m^{-2}\,\mu m^{-1}}$.
Jde o spektrální hustotu toku fotonů, nikoli o celkový počet fotonů v pásmu.

> **💻 Co dělá kód:** Převede energii fotonu z eV zpět na jouly násobením `e` a vydělí jí spektrální sluneční ozářenost `E0`. Vykreslí `E0` spolu se spektrální hustotou toku fotonů (za sekundu na m² a µm spektra) a ukáže tabulku.

In [ ]:
bands["photons_per_s_m2_um"] = bands.E0_W_m2_um / (bands.E_photon_eV * e)
fig, ax1 = plt.subplots(figsize=(9, 3.2))
ax1.plot(
    bands.lambda_nm, bands.E0_W_m2_um, "o-", c="tab:orange", label="E0 (W·m⁻²·µm⁻¹)"
)
ax2 = ax1.twinx()
ax2.plot(
    bands.lambda_nm,
    bands.photons_per_s_m2_um / 1e21,
    "s--",
    c="tab:blue",
    label="fotony (10²¹ s⁻¹·m⁻²·µm⁻¹)",
)
ax1.set(xlabel="vlnová délka (nm)", ylabel="výkon")
ax2.set(ylabel="počet fotonů", ylim=(0, None))
ax1.set_ylim(0)
fig.legend(loc="lower left", bbox_to_anchor=(0.08, 0.15))
plt.show()
bands[["E0_W_m2_um", "E_photon_eV", "photons_per_s_m2_um"]].style.format("{:.3g}")

> **✏️ Úkol 3.** Výkon E0 klesá od modré (B2) k SWIR 2 (B12) více než dvacetkrát. Klesá stejně i počet fotonů? Vysvětli rozdíl. V kterém pásmu dopadá na metr čtvereční nejvíc fotonů na mikrometr spektra a proč to není pásmo s nejvyšším E0?

*Tvoje odpověď:* 

## 5. Které pásmo uvidí křemíkový čip

Foton detektor uvolní elektron jen tehdy, když má aspoň energii zakázaného pásu materiálu. U křemíku je to asi 1,12 eV. Spočítáme mezní vlnovou délku a podíváme se, jak velká část citlivosti každého pásma leží pod ní.

> **💻 Co dělá kód:** Ze šířky zakázaného pásu `Eg` spočítá mezní vlnovou délku detektoru a pro každé pásmo podíl jeho spektrální citlivosti pod touto mezí. Ukáže tabulku.

In [ ]:
# Křemík 1,12; InGaAs asi 0,75; HgCdTe podle složení 0,1–0,5.
Eg = 1.12  # ◀ parametr: šířka zakázaného pásu v eV
lam_cut = h * c / (Eg * e) * 1e9
print(f"mezní vlnová délka: {lam_cut:.0f} nm")
share = {name: r[wl <= lam_cut].sum() / r.sum() for name, (wl, r) in srf.items()}
bands["share_detectable"] = pd.Series(share)
bands[["lambda_nm", "E_photon_eV", "share_detectable"]].style.format(
    {"lambda_nm": "{:.0f}", "E_photon_eV": "{:.2f}", "share_detectable": "{:.0%}"}
)

> **✏️ Úkol 4.** Která pásma Sentinelu-2 by křemíkový detektor zachytil celá, která jen zčásti a která vůbec? Nastav `Eg = 0.75` (InGaAs) a spusť buňku znovu. Co se změnilo? Sentinel-2 má pro pásma B10 až B12 detektory z HgCdTe: proč?

*Tvoje odpověď:* 

## 6. Termální záření a radar

Porovnáme fotony z termálního pásma (10,9 µm, jako Landsat 8 TIRS) a z radaru Sentinel-1 (5,405 GHz) s typickou tepelnou energií kmitů v detektoru při pokojové teplotě, k · T.

> **💻 Co dělá kód:** Spočítá energii fotonu pro ionizaci, modré a SWIR pásmo, termální pásmo a radar C a porovná ji s tepelnou energií k · T při teplotě detektoru `T`. Ukáže tabulku.

In [ ]:
T = 300  # ◀ parametr: teplota detektoru v kelvinech
kT = k * T / e
other = pd.DataFrame(
    {
        "lambda_m": [h * c / (10 * e), 490e-9, 1610e-9, 10.9e-6, c / 5.405e9],
    },
    index=[
        "ionizace (10 eV)",
        "B2 modrá",
        "B11 SWIR",
        "termální 10,9 µm",
        "radar C 5,405 GHz",
    ],
)
other["E_eV"] = h * c / other.lambda_m / e
other["E_over_kT"] = other.E_eV / kT
print(f"k·T při {T} K = {kT * 1000:.1f} meV")
other.style.format({"lambda_m": "{:.3g}", "E_eV": "{:.3g}", "E_over_kT": "{:.3g}"})

> **✏️ Úkol 5.** Termální foton má energii jen asi čtyřikrát větší než k · T, radarový foton je víc než tisíckrát slabší než k · T. Proč proto termální kamery s fotonovými detektory chladí a proč mikrobolometr a radarová anténa neměří jednotlivé fotony, ale ohřev, resp. elektrické pole vlny? Nastav `T = 77` (kapalný dusík) a porovnej.

*Tvoje odpověď:* 

> **✏️ Úkol 6.** Na jaké vlnové délce má foton energii 10 eV (přibližná hranice ionizace)? Do které oblasti spektra patří? Chemické vazby v molekulách mají energii v řádu několika eV. Vysvětli, proč jediný foton UV záření může poškodit DNA, zatímco silný radarový signál nese celkem víc energie, a přesto žádnou vazbu nerozbije ani nic neionizuje.

*Tvoje odpověď:* 

> **✏️ Rozšíření (nepovinné).** Spolužáci mohou mít notebook pro jinou družici řady Sentinel-2 (2A, 2B nebo 2C). Porovnejte střední vlnové délky pásem a hodnoty E0: o kolik nanometrů se pásma stejného jména liší a proč s tím musí počítat, kdo skládá časovou řadu ze snímků více družic?

*Tvoje odpověď:* 

---
### Zdroje dat a literatura

- Copernicus Sentinel-2 data 2025, ESA; metadata L1C z veřejného archivu Google Cloud (`gcp-public-data-sentinel-2`). Licence: [Legal notice on the use of Copernicus Sentinel Data](https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice), volné užití s uvedením zdroje.
- Konstanty h, c, e, k: v SI od roku 2019 přesně definované hodnoty, převzaté z `scipy.constants`; přehled CODATA 2022: [NIST](https://physics.nist.gov/cuu/pdf/wall_2022.pdf).
- Šířka zakázaného pásu křemíku 1,12 eV při 300 K: Sze, S. M., Ng, K. K. (2007). *Physics of Semiconductor Devices*, 3. vyd. Wiley.
- Frekvence radaru Sentinel-1 (pásmo C) 5,405 GHz: ESA, Sentinel-1 User Handbook.

### Jak citovat

OpenSkyLab (2026). Fotony v pásmech Sentinelu-2C [notebook, výukový materiál], verze 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/rope_and_photons_brno_cs.ipynb. Licence CC BY 4.0.

Text a kód notebooku: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data mají licence svých poskytovatelů (viz výše).

*Materiál připravil tým OpenSkyLab s pomocí generativní AI Claude (Anthropic), která pomáhala s návrhem kódu a textu. Za obsah odpovídají autoři.*